In [1]:
# checking if finbert runs properly
from transformers import pipeline

# Load FinBERT (first run downloads ~440 MB; subsequent runs use cache)
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# Three test sentences
sentences = [
    "Revenue grew 25% year over year and margins expanded significantly.",
    "We are facing significant headwinds and expect continued margin compression.",
    "The quarter ended on December 31, 2024.",
]

results = finbert(sentences)
for sent, res in zip(sentences, results):
    print(f"{res['label']:10s} ({res['score']:.3f})  |  {sent}")

d:\Projects\risk-radar\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\risk-radar\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dsbha\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this 

positive   (0.958)  |  Revenue grew 25% year over year and margins expanded significantly.
negative   (0.970)  |  We are facing significant headwinds and expect continued margin compression.
neutral    (0.899)  |  The quarter ended on December 31, 2024.


In [21]:
import re
import pandas as pd

def split_transcript(text: str) -> dict:
    """
    Split a raw earnings call transcript into prepared_remarks and qa sections.
    
    Returns dict with 3 keys:
        prepared_remarks: str
        qa: str
        format: 'standard' | 'interview' | 'qa_missing'
    """
    # --- Detect interview format FIRST (NFLX, TSLA 2021-Q3+) ---
    opening = text[:800].lower()
    is_interview = (
        "earnings interview" in opening
        or "our interviewer" in opening
        or "interviewer this quarter" in opening
        or "questions submitted" in opening
        or "shareholder questions" in opening
        or "q&a webcast" in opening
        or "question-and-answer webcast" in opening
    )
    
    if is_interview:
        end_match = re.search(r"Call participants:|Duration:", text)
        body = text[:end_match.start()].strip() if end_match else text.strip()
        body = re.sub(r"^Prepared Remarks:\s*\n", "", body)
        return {"prepared_remarks": "", "qa": body, "format": "interview"}
    
    # --- Standard format from here on ---
    
    # Cut 3: drop end-of-call metadata
    end_match = re.search(r"Call participants:", text)
    if end_match:
        text = text[:end_match.start()]
    else:
        end_match = re.search(r"Duration:", text)
        if end_match:
            text = text[:end_match.start()]
    
    # Cut 2: find Q&A boundary — flexible regex catches:
    #   "Questions and Answers:" / "Questions & Answers:" / "Question and Answer Session"
    qa_patterns = [
        r"\n\s*Questions?\s*(?:and|&)\s*Answers?\s*:\s*\n",
        r"\n\s*Question[-\s]and[-\s]Answer\s+Session\s*\n",
    ]
    qa_start = None
    prepared_end = None
    for pattern in qa_patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            qa_start = m.end()
            prepared_end = m.start()
            break
    
    # Fallback: no explicit Q&A header. Use "first question" as the anchor.
    if qa_start is None:
        twenty_pct = len(text) // 5
        fq_match = re.search(r"first question", text[twenty_pct:], re.IGNORECASE)
        if fq_match:
            fq_pos = twenty_pct + fq_match.start()
            lookback_start = max(0, fq_pos - 500)
            operator_match = None
            for m in re.finditer(r"\nOperator\n", text[lookback_start:fq_pos]):
                operator_match = m
            if operator_match:
                qa_start = lookback_start + operator_match.end()
                prepared_end = lookback_start + operator_match.start()
            else:
                qa_start = fq_pos
                prepared_end = fq_pos
    
    # Cut 1: skip past "Prepared Remarks:" header
    prep_header = re.search(r"Prepared Remarks:", text)
    prepared_start = prep_header.end() if prep_header else 0
    
    # Slice the sections
    if qa_start is not None:
        prepared_remarks = text[prepared_start:prepared_end].strip()
        qa = text[qa_start:].strip()
    else:
        prepared_remarks = text[prepared_start:].strip()
        qa = ""
    
    # Edge case: Q&A section is empty or near-empty (transcription failure)
    # Real Q&A sections are always 1000+ words. <50 = data is missing.
    if len(qa.split()) < 50:
        return {"prepared_remarks": prepared_remarks, "qa": "", "format": "qa_missing"}
    
    return {"prepared_remarks": prepared_remarks, "qa": qa, "format": "standard"}

In [22]:
# Re-apply the splitter on all 273 transcripts
df = pd.read_csv("../data/processed/transcripts_v2.csv")

split_results = df['transcript'].apply(split_transcript)
df['prepared_remarks'] = split_results.apply(lambda d: d['prepared_remarks'])
df['qa'] = split_results.apply(lambda d: d['qa'])
df['format'] = split_results.apply(lambda d: d['format'])
df['prep_words'] = df['prepared_remarks'].str.split().str.len()
df['qa_words'] = df['qa'].str.split().str.len()

# --- Diagnostic 1: format breakdown ---
print("=== FORMAT BREAKDOWN ===")
print(df['format'].value_counts())
print()

# --- Diagnostic 2: which tickers are in interview format ---
print("=== INTERVIEW format — which tickers/quarters ===")
intv = df[df['format']=='interview'][['ticker', 'quarter', 'qa_words']]
print(intv.to_string())
print()

# --- Diagnostic 3: standard format quality ---
std = df[df['format']=='standard'].copy()
std['prep_ratio'] = std['prep_words'] / (std['prep_words'] + std['qa_words']).clip(lower=1)

print(f"=== STANDARD format quality ({len(std)} transcripts) ===")
print(f"  Empty Q&A:                {(std['qa_words']==0).sum()}")
print(f"  Tiny prep (<500 words):   {(std['prep_words']<500).sum()}")
print(f"  Huge prep ratio (>0.85):  {(std['prep_ratio']>0.85).sum()}")
print()

# --- Diagnostic 4: remaining suspect rows (if any) ---
suspects = std[(std['qa_words']==0) | (std['prep_ratio']>0.85)]
print(f"=== REMAINING SUSPECTS in standard format ({len(suspects)}) ===")
if len(suspects) > 0:
    print(suspects[['ticker', 'quarter', 'prep_words', 'qa_words', 'prep_ratio']].round(2).to_string())
else:
    print("  None — all standard transcripts split cleanly.")

=== FORMAT BREAKDOWN ===
format
standard      251
interview      21
qa_missing      1
Name: count, dtype: int64

=== INTERVIEW format — which tickers/quarters ===
    ticker  quarter  qa_words
136   NFLX  2020-Q4      7689
137   NFLX  2021-Q1      8091
138   NFLX  2021-Q2      7340
139   NFLX  2021-Q3      7783
140   NFLX  2021-Q4      8332
141   NFLX  2022-Q1      8024
142   NFLX  2022-Q2      8372
143   NFLX  2022-Q3      8196
144   NFLX  2022-Q4      9120
220   TSLA  2019-Q3      9493
221   TSLA  2019-Q4      9668
222   TSLA  2020-Q1      9073
223   TSLA  2020-Q2      9213
224   TSLA  2020-Q4      9236
225   TSLA  2021-Q1      8913
226   TSLA  2021-Q2      8380
227   TSLA  2021-Q3      9480
228   TSLA  2021-Q4      9096
229   TSLA  2022-Q2      9507
230   TSLA  2022-Q3      8634
231   TSLA  2022-Q4      8977

=== STANDARD format quality (251 transcripts) ===
  Empty Q&A:                0
  Tiny prep (<500 words):   6
  Huge prep ratio (>0.85):  0

=== REMAINING SUSPECTS in standard 

In [23]:
import random
random.seed(42)  # reproducible

# Pick 5 random standard-format transcripts
std_indices = df[df['format']=='standard'].index.tolist()
sample = random.sample(std_indices, 5)

for idx in sample:
    row = df.iloc[idx]
    print(f"\n{'='*70}")
    print(f"  {row['ticker']} {row['quarter']}  |  prep={row['prep_words']}w, qa={row['qa_words']}w")
    print(f"{'='*70}")
    
    print("\n--- LAST 250 CHARS OF PREPARED REMARKS ---")
    print(row['prepared_remarks'][-250:])
    
    print("\n--- FIRST 250 CHARS OF Q&A ---")
    print(row['qa'][:250])


  PTON 2020-Q2  |  prep=2291w, qa=6659w

--- LAST 250 CHARS OF PREPARED REMARKS ---
ITDA ranges. For fiscal year 2020, we expect adjusted EBITDA in the range of negative $115 million to negative $95 million and an adjusted EBITDA margin of negative 6.8% at the midpoint.
I will now turn it over to the operator to take your questions.

--- FIRST 250 CHARS OF Q&A ---
Operator
Thank you. [Operator Instructions] Our first question comes from Doug Anmuth with J.P. Morgan.
Douglas Anmuth -- J.P. Morgan -- Analyst
Great. Thanks for taking the questions. Just two I wanted to ask. John, first, you talked more about the 

  AMZN 2022-Q3  |  prep=1857w, qa=3226w

--- LAST 250 CHARS OF PREPARED REMARKS ---
lling partners.
We remain heads-down focused on driving a fantastic customer experience, and we believe putting customers first is the only reliable way to create lasting value for shareholders. Thanks. And with that, let's move on to your questions.

--- FIRST 250 CHARS OF Q&A ---
Operator
[Ope

In [24]:
import nltk
nltk.download('punkt_tab')  # the sentence tokenizer data

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dsbha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [35]:
from nltk.tokenize import sent_tokenize

def chunk_text(text: str, target_words: int = 380, max_words: int = 380) -> list[str]:
    """
    Split text into chunks at sentence boundaries.
    Each chunk is ~target_words words, never exceeding max_words (unless a single
    sentence is longer than max_words, in which case that sentence stands alone).
    
    Returns list of chunk strings. Empty input returns [].
    """
    if not text or not text.strip():
        return []
    
    sentences = sent_tokenize(text)
    
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence_word_count = len(sentence.split())
        
        # Would adding this sentence exceed the hard cap?
        if current_word_count + sentence_word_count > max_words and current_chunk:
            # Close current chunk and start fresh with this sentence
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_word_count = sentence_word_count
        else:
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
        
        # If we've hit the target, close the chunk
        if current_word_count >= target_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_word_count = 0
    
    # Don't forget the last partial chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [36]:
# Pick a known transcript with both sections (AAPL 2019-Q3 — the one we know well)
sample = df[(df['ticker']=='AAPL') & (df['quarter']=='2019-Q3')].iloc[0]

prep_chunks = chunk_text(sample['prepared_remarks'])
qa_chunks = chunk_text(sample['qa'])

print(f"=== AAPL 2019-Q3 ===")
print(f"Prepared remarks: {sample['prep_words']} words → {len(prep_chunks)} chunks")
print(f"Q&A:              {sample['qa_words']} words → {len(qa_chunks)} chunks")
print()
print(f"=== Chunk word counts (prep) ===")
for i, c in enumerate(prep_chunks):
    wc = len(c.split())
    flag = "⚠️" if wc > 450 else "✓"
    print(f"  Chunk {i}: {wc} words  {flag}")
print()
print(f"=== Chunk word counts (qa) ===")
for i, c in enumerate(qa_chunks):
    wc = len(c.split())
    flag = "⚠️" if wc > 450 else "✓"
    print(f"  Chunk {i}: {wc} words  {flag}")
print()
print(f"=== First chunk of prep (first 300 chars) ===")
print(prep_chunks[0][:300])
print(f"\n=== Last chunk of qa (first 300 chars) ===")
print(qa_chunks[-1][:300])

=== AAPL 2019-Q3 ===
Prepared remarks: 4105 words → 12 chunks
Q&A:              4156 words → 12 chunks

=== Chunk word counts (prep) ===
  Chunk 0: 378 words  ✓
  Chunk 1: 361 words  ✓
  Chunk 2: 376 words  ✓
  Chunk 3: 372 words  ✓
  Chunk 4: 366 words  ✓
  Chunk 5: 366 words  ✓
  Chunk 6: 351 words  ✓
  Chunk 7: 373 words  ✓
  Chunk 8: 367 words  ✓
  Chunk 9: 380 words  ✓
  Chunk 10: 374 words  ✓
  Chunk 11: 42 words  ✓

=== Chunk word counts (qa) ===
  Chunk 0: 376 words  ✓
  Chunk 1: 368 words  ✓
  Chunk 2: 378 words  ✓
  Chunk 3: 347 words  ✓
  Chunk 4: 371 words  ✓
  Chunk 5: 356 words  ✓
  Chunk 6: 367 words  ✓
  Chunk 7: 376 words  ✓
  Chunk 8: 350 words  ✓
  Chunk 9: 378 words  ✓
  Chunk 10: 352 words  ✓
  Chunk 11: 137 words  ✓

=== First chunk of prep (first 300 chars) ===
Operator
Good day and welcome to the Apple Incorporated Third Quarter Fiscal Year 2019 Earnings Conference Call. [Operator Instructions]. At this time for opening remarks and introductions, I would like to

In [37]:
# Estimate total chunks across all 273 transcripts
prep_chunks_total = 0
qa_chunks_total = 0

for _, row in df.iterrows():
    prep_chunks_total += len(chunk_text(row['prepared_remarks']))
    qa_chunks_total += len(chunk_text(row['qa']))

total = prep_chunks_total + qa_chunks_total
print(f"Total chunks to score:")
print(f"  Prepared remarks: {prep_chunks_total:,}")
print(f"  Q&A:              {qa_chunks_total:,}")
print(f"  TOTAL:            {total:,}")
print()
print(f"At ~1-2 sec per chunk on CPU, this run will take ~{total*1.5/60:.0f} minutes")

Total chunks to score:
  Prepared remarks: 2,651
  Q&A:              4,619
  TOTAL:            7,270

At ~1-2 sec per chunk on CPU, this run will take ~182 minutes


In [38]:
import time
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    top_k=None,           # return all 3 scores
    device=-1,            # CPU
    truncation=True,      # safety: cut at 512 tokens
    max_length=512,
)

sample_chunks = chunk_text(df.iloc[0]['prepared_remarks'])[:20]
print(f"Timing test: {len(sample_chunks)} chunks")

start = time.time()
results = finbert(sample_chunks, batch_size=4)
elapsed = time.time() - start

print(f"Elapsed: {elapsed:.1f}s")
print(f"Per-chunk: {elapsed/len(sample_chunks):.2f}s")
print(f"Projected total for 6,852 chunks: {elapsed/len(sample_chunks)*6852/60:.0f} minutes")
print()
print("Example result format:")
print(results[0])

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18274.45it/s]


Timing test: 12 chunks
Elapsed: 5.6s
Per-chunk: 0.47s
Projected total for 6,852 chunks: 53 minutes

Example result format:
[{'label': 'positive', 'score': 0.8990704417228699}, {'label': 'neutral', 'score': 0.08864070475101471}, {'label': 'negative', 'score': 0.012288917787373066}]


In [31]:
print("=== SANITY CHECK 1: Are all chunks valid? ===")
bad_chunks = []
empty_sections = []

for idx, row in df.iterrows():
    for section in ['prepared_remarks', 'qa']:
        chunks = chunk_text(row[section])
        if len(chunks) == 0:
            empty_sections.append((row['ticker'], row['quarter'], section, row['format']))
            continue
        for i, c in enumerate(chunks):
            if not c or not c.strip():
                bad_chunks.append((row['ticker'], row['quarter'], section, i))

print(f"Empty sections (expected for interview-format prep, qa_missing): {len(empty_sections)}")
print(f"Bad chunks (whitespace-only inside a non-empty section): {len(bad_chunks)}")

if bad_chunks:
    print("\n⚠️ BAD CHUNKS — investigate:")
    for b in bad_chunks[:5]:
        print(f"  {b}")

# Empty sections — confirm they match our format tagging
print("\nEmpty section breakdown:")
empty_df = pd.DataFrame(empty_sections, columns=['ticker','quarter','section','format'])
print(empty_df.groupby(['format', 'section']).size())

=== SANITY CHECK 1: Are all chunks valid? ===
Empty sections (expected for interview-format prep, qa_missing): 22
Bad chunks (whitespace-only inside a non-empty section): 0

Empty section breakdown:
format      section         
interview   prepared_remarks    21
qa_missing  qa                   1
dtype: int64


In [39]:
print("=== SANITY CHECK 2: How many chunks will get truncated? ===")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# Sample 200 random chunks from across the dataset to estimate
import random
random.seed(0)
all_chunks = []
for idx, row in df.iterrows():
    all_chunks.extend(chunk_text(row['prepared_remarks']))
    all_chunks.extend(chunk_text(row['qa']))

print(f"Total chunks: {len(all_chunks)}")

sample = random.sample(all_chunks, 200)
token_counts = [len(tokenizer.encode(c)) for c in sample]

over_512 = sum(1 for t in token_counts if t > 512)
print(f"Sample of 200 chunks:")
print(f"  Median tokens: {sorted(token_counts)[100]}")
print(f"  Max tokens: {max(token_counts)}")
print(f"  Over 512 (will be truncated): {over_512}/200 = {over_512/2:.1f}%")

=== SANITY CHECK 2: How many chunks will get truncated? ===


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 7270
Sample of 200 chunks:
  Median tokens: 469
  Max tokens: 558
  Over 512 (will be truncated): 7/200 = 3.5%


In [33]:
print("=== SANITY CHECK 3: Score one full call ===")
import time

def score_chunks(chunks, batch_size=8):
    """Score list of chunks, return list of dicts with pos/neu/neg probs."""
    if len(chunks) == 0:
        return []
    results = finbert(chunks, batch_size=batch_size)
    scored = []
    for r in results:
        # r is a list of 3 dicts: [{'label':'positive','score':...}, ...]
        d = {item['label']: item['score'] for item in r}
        scored.append(d)
    return scored

def aggregate(scored_chunks, prefix):
    """Aggregate chunk-level scores to section-level features."""
    if len(scored_chunks) == 0:
        return {
            f"{prefix}_pos_mean": None, f"{prefix}_neu_mean": None, f"{prefix}_neg_mean": None,
            f"{prefix}_pos_share": None, f"{prefix}_neg_share": None,
            f"{prefix}_n_chunks": 0,
        }
    pos_scores = [c['positive'] for c in scored_chunks]
    neu_scores = [c['neutral'] for c in scored_chunks]
    neg_scores = [c['negative'] for c in scored_chunks]
    # Winner-takes-all label per chunk
    winners = [max(c, key=c.get) for c in scored_chunks]
    return {
        f"{prefix}_pos_mean": sum(pos_scores)/len(pos_scores),
        f"{prefix}_neu_mean": sum(neu_scores)/len(neu_scores),
        f"{prefix}_neg_mean": sum(neg_scores)/len(neg_scores),
        f"{prefix}_pos_share": sum(1 for w in winners if w=='positive') / len(winners),
        f"{prefix}_neg_share": sum(1 for w in winners if w=='negative') / len(winners),
        f"{prefix}_n_chunks": len(scored_chunks),
    }

# Dress-rehearsal on one call
row = df.iloc[0]  # AAPL 2019-Q3
print(f"Test call: {row['ticker']} {row['quarter']}")

start = time.time()
prep_chunks = chunk_text(row['prepared_remarks'])
qa_chunks = chunk_text(row['qa'])
prep_scored = score_chunks(prep_chunks)
qa_scored = score_chunks(qa_chunks)
elapsed = time.time() - start

features = {**aggregate(prep_scored, 'prep'), **aggregate(qa_scored, 'qa')}

print(f"Elapsed: {elapsed:.1f}s for {len(prep_chunks)+len(qa_chunks)} chunks")
print(f"Features extracted:")
for k, v in features.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

=== SANITY CHECK 3: Score one full call ===
Test call: AAPL 2019-Q3
Elapsed: 11.6s for 22 chunks
Features extracted:
  prep_pos_mean: 0.660
  prep_neu_mean: 0.283
  prep_neg_mean: 0.057
  prep_pos_share: 0.636
  prep_neg_share: 0.000
  prep_n_chunks: 11
  qa_pos_mean: 0.255
  qa_neu_mean: 0.713
  qa_neg_mean: 0.032
  qa_pos_share: 0.182
  qa_neg_share: 0.000
  qa_n_chunks: 11


In [34]:
print("=== SANITY CHECK 4: Disk write ===")
import os
test_path = "../data/processed/finbert_test_write.csv"
test_df = pd.DataFrame([{"test": 1, "ok": True}])
test_df.to_csv(test_path, index=False)

if os.path.exists(test_path):
    print(f"✅ Write OK: {test_path}")
    os.remove(test_path)
    print("✅ Cleanup OK")
else:
    print("❌ Could not write — check folder exists and permissions")

=== SANITY CHECK 4: Disk write ===
✅ Write OK: ../data/processed/finbert_test_write.csv
✅ Cleanup OK


In [40]:
import os
import time
import pandas as pd
from tqdm import tqdm

OUTPUT_PATH = "../data/processed/finbert_scores.csv"
TMP_PATH = "../data/processed/finbert_scores_tmp.csv"
SAVE_EVERY_N = 10  # save partial results every N calls

# --- Resume logic: check what's already done ---
done_keys = set()
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_csv(OUTPUT_PATH)
    done_keys = set(zip(existing['ticker'], existing['quarter']))
    results_so_far = existing.to_dict('records')
    print(f"📂 Resuming — found {len(done_keys)} calls already scored")
else:
    results_so_far = []
    print("🆕 Fresh run — no partial results found")

# --- Helpers (re-defined for safety, in case kernel was restarted) ---
def score_chunks(chunks, batch_size=8):
    if len(chunks) == 0:
        return []
    results = finbert(chunks, batch_size=batch_size)
    return [{item['label']: item['score'] for item in r} for r in results]

def aggregate(scored, prefix):
    if len(scored) == 0:
        return {
            f"{prefix}_pos_mean": None, f"{prefix}_neu_mean": None, f"{prefix}_neg_mean": None,
            f"{prefix}_pos_share": None, f"{prefix}_neg_share": None,
            f"{prefix}_n_chunks": 0,
        }
    pos = [c['positive'] for c in scored]
    neu = [c['neutral']  for c in scored]
    neg = [c['negative'] for c in scored]
    winners = [max(c, key=c.get) for c in scored]
    return {
        f"{prefix}_pos_mean":  sum(pos)/len(pos),
        f"{prefix}_neu_mean":  sum(neu)/len(neu),
        f"{prefix}_neg_mean":  sum(neg)/len(neg),
        f"{prefix}_pos_share": sum(1 for w in winners if w=='positive') / len(winners),
        f"{prefix}_neg_share": sum(1 for w in winners if w=='negative') / len(winners),
        f"{prefix}_n_chunks":  len(scored),
    }

def atomic_save(records, output_path, tmp_path):
    """Write to tmp file first, then rename. Never leaves a corrupt file."""
    pd.DataFrame(records).to_csv(tmp_path, index=False)
    os.replace(tmp_path, output_path)  # atomic on Windows + Linux

# --- The main loop ---
to_process = [
    row for _, row in df.iterrows()
    if (row['ticker'], row['quarter']) not in done_keys
]
print(f"🚀 Starting — {len(to_process)} calls to process")

run_start = time.time()

for i, row in enumerate(tqdm(to_process, desc="Scoring calls")):
    call_start = time.time()
    
    prep_chunks = chunk_text(row['prepared_remarks'])
    qa_chunks   = chunk_text(row['qa'])
    
    prep_scored = score_chunks(prep_chunks)
    qa_scored   = score_chunks(qa_chunks)
    
    record = {
        'ticker':       row['ticker'],
        'quarter':      row['quarter'],
        'date_parsed':  row['date_parsed'],
        'format':       row['format'],
        **aggregate(prep_scored, 'prep'),
        **aggregate(qa_scored,   'qa'),
        'call_seconds': round(time.time() - call_start, 1),
    }
    results_so_far.append(record)
    
    # Incremental save
    if (i + 1) % SAVE_EVERY_N == 0:
        atomic_save(results_so_far, OUTPUT_PATH, TMP_PATH)

# Final save
atomic_save(results_so_far, OUTPUT_PATH, TMP_PATH)

total_min = (time.time() - run_start) / 60
print(f"\n✅ Complete — {len(results_so_far)} calls scored, total {total_min:.1f} min")
print(f"📁 Saved to {OUTPUT_PATH}")

🆕 Fresh run — no partial results found
🚀 Starting — 273 calls to process


Scoring calls: 100%|██████████| 273/273 [1:01:50<00:00, 13.59s/it]


✅ Complete — 273 calls scored, total 61.8 min
📁 Saved to ../data/processed/finbert_scores.csv
